1) Positiv/Negativ-Klassifikator 

Überwachte Klassifikation: Snippet + Deployment-KPIs → Label `1` (negativ/mutiert) und `0` (positiv).

Dient als Proof of Concept, ob die Snippets anhand der kpi Metriken in positiv und negatibeispiele eingeteilt werden können. 


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit



RANDOM_STATE = 42 

TRAINING_PROJECTS = ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11"]
EVAL_PROJECTS = ["P12", "P13"]
ENVS = ["low", "medium", "high", "extreme", "prod"]

/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
KPI_CSV_DATA = "../export_kpis/all_projects_kpis_new.csv"
df = pd.read_csv(KPI_CSV_DATA)
is_neg = df["variant"].str.endswith("_neg")
neg = is_neg.sum()
pos = (~is_neg).sum()
print("Zeile insgesammt", len(df))
print("Davon positive Datensaetze :", pos)
print("Negative Datensaetze :", neg)

if neg == 0 or pos == 0:
    raise SystemExit(
        "Entweder keine positiven oder negativen datensaetze vorhanden"
    ) 

2) Label & Feature Matrix 

- label is_neg -> negativ Endung -> wird zu 1 (negativ)
- base_project : Projekt ohne _neg endung  -> 0 (positiv)
- pair-id : Pärchen Gruppen -> pos+neg Code bleiben zusammen
 

In [ ]:
df["is_neg"] = df["variant"].str.endswith("_neg").astype(int)
df["project_id"] = df["variant"].str.replace("_neg","", regex=False)
df["pair_id"] = df["target_method"]

3) Training und Evaluation Datenset aufteilen 


In [ ]:
train_df = df[df["project_id"].isin(TRAINING_PROJECTS)]
test_df = df[df["project_is"].isin(EVAL_PROJECTS)]

shufflesplit = GroupShuffleSplit (n_splits=1,test_size=0.25,random_state=RANDOM_STATE)
train_idx,val_idx = next(shufflesplit.split(train_df, groups= train_df["pair_id"]))

train_df = train_df.iloc[train_idx].copy()
val_df = train_df.iloc[val_idx].copy()

print("Train-Projekte:", sorted(train_df["project_id"].unique()))
print("Val-Projekte:  ", sorted(val_df["project_id"].unique()))
print("Test (held-out):", sorted(test_df["project_id"].unique()))
print("n train / val / test:", len(train_df), len(val_df), len(test_df))